# 🧑‍🍳 Neosantara AI Cookbook: RAG with ChromaDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/rag-chromadb.ipynb)

RAG with `llama-3.3-nemotron-super-49b-v1.5`.

In [ ]:
# Install dependencies\n!pip install openai\n\nimport os\nfrom getpass import getpass\n\ntry:\n    from google.colab import userdata\n    # Try to get from Colab Secrets first\n    NEOSANTARA_API_KEY = userdata.get("NEOSANTARA_API_KEY")\n    if not NEOSANTARA_API_KEY:\n        raise ValueError("Secret not found")\n    os.environ["NEOSANTARA_API_KEY"] = NEOSANTARA_API_KEY\n    print("✅ API Key loaded from Google Colab Secrets!")\nexcept (ImportError, Exception):\n    # Fallback to manual input if not found or not in Colab\n    if "NEOSANTARA_API_KEY" not in os.environ:\n        print("🔑 API Key not found in Secrets. Please paste it below:")\n        os.environ["NEOSANTARA_API_KEY"] = getpass("Neosantara API Key: ")\n    else:\n        print("✅ API Key loaded from environment!")

In [ ]:
from openai import OpenAI
import os
import chromadb

client = OpenAI(
    api_key=os.getenv("NEOSANTARA_API_KEY"),
    base_url="https://api.neosantara.xyz/v1"
)

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model="nusa-embedding-0001")
    return response.data[0].embedding

db = chromadb.Client()
collection = db.create_collection(name="docs")
collection.add(documents=["Neosantara is an AI gateway."], embeddings=[get_embedding("Neosantara is an AI gateway.")], ids=["id1"])

results = collection.query(query_embeddings=[get_embedding("What is Neosantara?")], n_results=1)
response = client.chat.completions.create(
    model="llama-3.3-nemotron-super-49b-v1.5",
    messages=[{"role": "user", "content": f"Context: {results['documents'][0][0]}\nQuestion: What is Neosantara?"}]
)
print(response.choices[0].message.content)